# Structural Analysis with Identified VARs

This tutorial walks through the complete structural VAR pipeline: fitting a model, applying Cholesky identification, and computing impulse responses, forecast error variance decompositions, and historical decompositions.

In [1]:
import numpy as np
import pandas as pd

from litterman import VAR, VARData
from litterman.identification import Cholesky
from litterman.samplers import NUTSSampler

## Simulate a VAR(1) process

In [2]:
rng = np.random.default_rng(42)
T = 200
y = np.zeros((T, 3))
for t in range(1, T):
    y[t, 0] = 0.6 * y[t - 1, 0] + rng.standard_normal() * 0.1
    y[t, 1] = 0.3 * y[t - 1, 0] + 0.5 * y[t - 1, 1] + rng.standard_normal() * 0.1
    y[t, 2] = 0.2 * y[t - 1, 1] + 0.4 * y[t - 1, 2] + rng.standard_normal() * 0.1

index = pd.date_range("2000-01-01", periods=T, freq="QS")
data = VARData(endog=y, endog_names=["gdp", "inflation", "rate"], index=index)

## Fit the model

In [3]:
sampler = NUTSSampler(draws=500, tune=500, chains=2, cores=1, random_seed=42)
fitted = VAR(lags=2, prior="minnesota").fit(data, sampler=sampler)
fitted

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, B, sigma_sd, tril_offdiag]


/Users/thomaspinder/Developer/litterman/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 2 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


FittedVAR(n_lags=2, n_vars=3, n_draws=500, n_chains=2)

## Apply Cholesky identification

The ordering determines the causal structure. Variables listed first are "most exogenous" — they are not contemporaneously affected by variables listed later.

In [4]:
identified = fitted.set_identification_strategy(
    Cholesky(ordering=["gdp", "inflation", "rate"])
)
identified

IdentifiedVAR(n_lags=2, n_vars=3, n_draws=500, n_chains=2)

## Impulse response functions

How does a one-standard-deviation structural shock propagate through the system?

In [5]:
irf = identified.impulse_response(horizon=20)
irf.median()

,0,1,2,3,4,5,6,7,8
0,0.102669,0.000000,0.000000,0.003412,0.099933,0.000000,-8.146595e-03,6.183652e-03,9.448409e-02
1,0.067115,0.000952,0.004327,0.012213,0.067399,-0.002708,-8.394381e-03,7.500920e-03,5.202568e-02
2,0.040002,0.001894,0.005441,0.017345,0.037661,-0.003038,-7.458884e-03,7.522230e-03,2.559837e-02
3,0.023711,0.002193,0.004655,0.016468,0.020094,-0.002075,-5.410997e-03,5.886091e-03,1.212313e-02
4,0.013922,0.002004,0.003374,0.013084,0.010592,-0.000970,-3.371212e-03,4.013163e-03,5.531959e-03
5,0.008238,0.001527,0.002249,0.009401,0.005687,-0.000243,-1.772686e-03,2.422388e-03,2.465598e-03
6,0.004935,0.001096,0.001394,0.006468,0.003087,0.000143,-8.778185e-04,1.411154e-03,1.105794e-03
7,0.003013,0.000766,0.000866,0.004278,0.001747,0.000264,-3.415371e-04,7.877351e-04,4.909105e-04
8,0.001865,0.000517,0.000533,0.002820,0.001001,0.000280,-1.162985e-04,4.373556e-04,2.062655e-04
9,0.001121,0.000332,0.000325,0.001803,0.000596,0.000235,-1.166182e-05,2.418048e-04,9.047471e-05


## Forecast error variance decomposition

What fraction of each variable's forecast error variance is explained by each structural shock?

In [6]:
fevd = identified.fevd(horizon=20)
fevd.median()

,0,1,2,3,4,5,6,7,8
0,1.000000,0.000000,0.000000,0.002868,0.997132,0.000000,0.007394,0.004884,0.982910
1,0.997471,0.000408,0.001335,0.012691,0.985687,0.000765,0.012541,0.008989,0.972650
2,0.993750,0.001277,0.003078,0.029569,0.965970,0.001975,0.017968,0.014039,0.962726
3,0.990542,0.002071,0.004407,0.044625,0.948760,0.002776,0.020787,0.016565,0.954881
4,0.988820,0.002701,0.005135,0.053983,0.938186,0.003213,0.021923,0.018096,0.951144
5,0.987822,0.003113,0.005589,0.059765,0.931475,0.003553,0.022374,0.018733,0.949950
6,0.987348,0.003399,0.005799,0.062594,0.928641,0.003716,0.022749,0.019051,0.949398
7,0.987144,0.003474,0.005887,0.063327,0.927112,0.003832,0.022830,0.019145,0.949018
8,0.987100,0.003508,0.005906,0.063792,0.926629,0.003913,0.022914,0.019163,0.948892
9,0.987078,0.003530,0.005916,0.064281,0.926333,0.003960,0.022922,0.019167,0.948825


## Historical decomposition

What was the contribution of each structural shock to each variable at each point in time?

In [7]:
hd = identified.historical_decomposition()
hd.median()

,0,1,2
0,0.090855,0.000000,0.000000
1,0.002858,-0.165112,0.000000
2,-0.007009,-0.010272,-0.137365
3,0.015781,0.000000,0.000000
4,0.000296,0.025985,0.000000
...,...,...,...
589,0.000850,0.167045,0.000000
590,-0.002316,0.010296,-0.114864
591,-0.120466,0.000000,0.000000
592,-0.003765,-0.061419,0.000000
